# Chronos-T5 Model

In [ ]:
import pandas as pd
import numpy as np
from chronos import BaseChronosPipeline
import os
import joblib
import torch

# Define Dataset configuration to handle different datasets
class DatasetConfig:
    def __init__(self, name, csv_path, window_size, base_model):
        self.name = name
        self.csv_path = csv_path
        self.window_size = window_size
        self.base_model = base_model

Define Chronos-T5 manager to generate synthetic/forecasting datasets based on specific configurations.

In [ ]:
class ChronosT5Manager():
    def __init__(self, config, scaler_path):
        self.config = config
        self.scaler = joblib.load(scaler_path)

        self.pipeline = BaseChronosPipeline.from_pretrained(
            self.config.base_model,
            device_map="cuda",
            dtype=torch.bfloat16
        )

        self.df = pd.read_csv(self.config.csv_path)
       
        self.numeric_columns = self.df.columns
        

    def generate(self, prediction_length=None, mode="synthetic"):
        print(f'Generating samples for {self.config.name} ...')
        if prediction_length is None:
            prediction_length = len(self.df)
            
        if mode == "forecast":
            real_seed_df = self.df.iloc[-self.config.window_size:]
        else:
            real_seed_df = self.df.iloc[:self.config.window_size]


        chunk_size = 64
        forecasts = {}

        if mode == 'synthetic':
            prediction_length -= self.config.window_size
        
        for col_idx, col_name in enumerate(self.df.columns):
            print(f'Generating {col_name} ...')
            current_context = torch.tensor(real_seed_df.iloc[:, col_idx].values).unsqueeze(0)
            
            generated_sequence = []
            steps_remaining = prediction_length
            
            while steps_remaining > 0:
                this_step_len = min(chunk_size, steps_remaining)
                window_input = current_context[:, -self.config.window_size:]
                
                forecast = self.pipeline.predict(
                    window_input,                  
                    prediction_length=this_step_len,
                    num_samples=20 
                )

                if mode == "synthetic":
                    random_sample_idx = torch.randint(0, forecast.shape[1], (1,)).item()
                    selected_pred = forecast[0, random_sample_idx, :] 
                else:
                    selected_pred = torch.median(forecast[0], dim=0).values
                
                selected_pred = selected_pred.cpu()
                generated_sequence.extend(selected_pred.numpy())
                
                selected_pred_tensor = selected_pred.unsqueeze(0)
                current_context = torch.cat((current_context, selected_pred_tensor), dim=1)

                steps_remaining -= this_step_len
            
            if mode == "synthetic":
                forecasts[col_name] = real_seed_df.iloc[:, col_idx].tolist() + generated_sequence
            else:
                forecasts[col_name] = generated_sequence

        df_raw_forecasts = pd.DataFrame(forecasts)
        df_numeric_part = df_raw_forecasts[self.numeric_columns]
        np_numeric_inverse = self.scaler.inverse_transform(df_numeric_part)
        df_final_numeric = pd.DataFrame(np_numeric_inverse, columns=self.numeric_columns)
        df_final = df_final_numeric
        
        
        if mode == "synthetic":
            output_path = self.config.csv_path.replace(
                "numerical.csv",
                "chronos_t5_synthetic.csv"
            )
        else:
            output_path = self.config.csv_path.replace(
                "numerical.csv",
                "chronos_t5_forecasting.csv"
            )
        df_final.to_csv(output_path, index=False)
        print(f"Done! Saved to {output_path}")
        # return df_final
       

Generate synthetic and forecasting data for different datasets with specific window size.

In [3]:
BASE_MODEL = "amazon/chronos-t5-base"

WINDOW_SIZE_DAILY = 30
WINDOW_SIZE_HOURLY = 48
WINDOW_SIZE_MINUTELY = 80

stock_test_len = len(pd.read_csv("./data/GOOGL_STOCKS/GOOGL_STOCKS_test.csv"))
occupancy_test_len = len(pd.read_csv("./data/OccupancyDetection/OccupancyDetection_test.csv"))
power_test_len = len(pd.read_csv( "./data/PowerConsumption/PowerConsumption_test.csv"))
traffic_test_len = len(pd.read_csv( "./data/TrafficVolume/TrafficVolume_test.csv"))


# Google Stocks Data
cfg_stocks = DatasetConfig("GOOGL_Stocks", "./data/GOOGL_STOCKS/GOOGL_STOCKS_numerical.csv", WINDOW_SIZE_DAILY, BASE_MODEL)
chronos_t5_stocks = ChronosT5Manager(cfg_stocks, './scaler/GOOGL_STOCKS_scaler.pkl')
chronos_t5_stocks.generate()
chronos_t5_stocks.generate(prediction_length=stock_test_len, mode="forecast")

# Occupancy Detection Data
cfg_occupancy = DatasetConfig("OccupancyDetection", "./data/OccupancyDetection/OccupancyDetection_numerical.csv", WINDOW_SIZE_MINUTELY, BASE_MODEL)
chronos_t5_occupancy = ChronosT5Manager(cfg_occupancy, './scaler/OccupancyDetection_scaler.pkl')
chronos_t5_occupancy.generate()
chronos_t5_occupancy.generate(prediction_length=occupancy_test_len, mode="forecast")

# Power Consumption Data
cfg_power = DatasetConfig("PowerConsumption", "./data/PowerConsumption/PowerConsumption_numerical.csv", WINDOW_SIZE_MINUTELY, BASE_MODEL)
chronos_t5_power = ChronosT5Manager(cfg_power, './scaler/PowerConsumption_scaler.pkl')
chronos_t5_power.generate()
chronos_t5_power.generate(prediction_length=power_test_len, mode="forecast")

# Traffic Volume Data
cfg_traffic = DatasetConfig("TrafficVolume", "./data/TrafficVolume/TrafficVolume_numerical.csv", WINDOW_SIZE_HOURLY, BASE_MODEL)
chronos_t5_traffic = ChronosT5Manager(cfg_traffic, './scaler/TrafficVolume_scaler.pkl')
chronos_t5_traffic.generate()
chronos_t5_traffic.generate(prediction_length=traffic_test_len, mode="forecast")


Generating samples for GOOGL_Stocks ...
Generating Open ...
Generating High ...
Generating Low ...
Generating Close ...
Generating Adj Close ...
Generating Volume ...
Done! Saved to ./data/GOOGL_STOCKS/GOOGL_STOCKS_chronos_t5_forecasting.csv
Generating samples for OccupancyDetection ...
Generating Temperature ...
Generating Humidity ...
Generating Light ...
Generating CO2 ...
Generating HumidityRatio ...
Generating Occupancy ...
Done! Saved to ./data/OccupancyDetection/OccupancyDetection_chronos_t5_forecasting.csv
Generating samples for PowerConsumption ...
Generating Temperature ...
Generating Humidity ...
Generating Wind_Speed ...
Generating general_diffuse_flows ...
Generating diffuse_flows ...
Generating Zone_1_Power_Consumption ...
Generating Zone_2_Power_Consumption ...
Generating Zone_3_Power_Consumption ...
Done! Saved to ./data/PowerConsumption/PowerConsumption_chronos_t5_forecasting.csv
Generating samples for TrafficVolume ...
Generating temp ...
Generating rain_1h ...
Genera